# 3.2 — Grant and Revoke RBAC & Privileges

**Exam domain:** Gen AI Governance · **Weight:** 28%

## The problem this solves

A data scientist builds an agent that answers questions over support tickets and the product
catalogue. It works perfectly in her session. She hands it to the BI team and every question comes
back empty. Nothing is broken — she has privileges on the search service, the semantic view and the
warehouse, and they have privileges on the agent and nothing else.

Cortex access is not one switch. It is a small stack of grants, and each feature adds its own object
privileges on top. This notebook is about knowing exactly which grants a given task needs, so you
stop guessing and stop over-granting.

## What you will be able to do

- Name the two grants every Cortex AI function call requires, and pick the right database role for a role's job
- Add the object grants that Cortex Search, Cortex Analyst and Cortex Agents each need on top
- Revoke access cleanly and prove what a role still holds
- Explain why a pipeline works interactively and produces nothing on a schedule
- Read `SHOW GRANTS` and `ACCOUNT_USAGE.GRANTS_TO_ROLES` to audit an account's Cortex posture

## Before you start

- Notebook 3.1, for the five gates and the model-access mechanisms
- `ACCOUNTADMIN`, or a role holding `MANAGE GRANTS`
- The `GENAI_STUDY` database from the repo setup script

📖 **Snowflake documentation for this notebook**
- [SNOWFLAKE database roles](https://docs.snowflake.com/en/sql-reference/snowflake-db-roles)
- [AISQL privileges and model access](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)
- [Cortex Analyst](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-analyst)
- [Cortex Search overview](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview)
- [Create and manage agents](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-manage)
- [Querying semantic views](https://docs.snowflake.com/en/user-guide/views-semantic/querying)


---
## The rule to memorise

> Calling a Cortex AI function needs **two** things: the account privilege
> `USE AI FUNCTIONS ON ACCOUNT` (or a per-function `USE AI FUNCTION <name>` grant), **and one of** the
> Cortex database roles below.

A **database role** is a bundle of privileges that lives inside a database. The Cortex ones live in
the `SNOWFLAKE` system database. You grant a database role to one of your own account roles — never
to a user directly — and then grant that account role to the user.

| Database role | Grants access to | `PUBLIC` by default? |
|---|---|---|
| `SNOWFLAKE.CORTEX_USER` | Cortex AI functions **and** Cortex services | **Yes** |
| `SNOWFLAKE.AI_FUNCTIONS_USER` | Scalar AI functions only — excludes `AI_AGG` and `AI_SUMMARIZE_AGG`, and all Cortex services | No |
| `SNOWFLAKE.CORTEX_EMBED_USER` | `AI_EMBED`, `AI_MULTI_EMBED`, `EMBED_TEXT_768`, `EMBED_TEXT_1024` | No |
| `SNOWFLAKE.CORTEX_ANALYST_USER` | Cortex Analyst only | No |
| `SNOWFLAKE.CORTEX_AGENT_USER` | Cortex Agents API only | No |
| `SNOWFLAKE.CORTEX_REST_API_USER` | The Cortex REST API only, without the rest of Cortex | No |
| `SNOWFLAKE.COPILOT_USER` | Cortex Code in Snowsight | **Yes** |

```sql
GRANT DATABASE ROLE SNOWFLAKE.<cortex_role> TO ROLE <account_role>;
REVOKE DATABASE ROLE SNOWFLAKE.<cortex_role> FROM ROLE <account_role>;
```

**Two things that catch people out**

- Only `ACCOUNTADMIN`, or a role with `MANAGE GRANTS`, can grant these database roles.
- Cortex roles cover the *function*. They never grant access to your *data*. `USAGE` on the database
  and schema, `SELECT` on the tables, `READ` on the stages and `USAGE` on the warehouse are always
  separate grants.

→ [More on the SNOWFLAKE database roles](https://docs.snowflake.com/en/sql-reference/snowflake-db-roles)
→ [More on AISQL privileges](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)


In [ ]:
%%sql
-- Functional roles for the GENAI_STUDY project
CREATE ROLE IF NOT EXISTS GENAI_ADMIN;     -- everything
CREATE ROLE IF NOT EXISTS GENAI_ANALYST;   -- AI functions + Analyst, no Agents
CREATE ROLE IF NOT EXISTS GENAI_EMBED_RO;  -- embeddings only (nightly vector refresh)

-- Every one of them needs the ACCOUNT PRIVILEGE as well as a database role
GRANT USE AI FUNCTIONS ON ACCOUNT TO ROLE GENAI_ADMIN;
GRANT USE AI FUNCTIONS ON ACCOUNT TO ROLE GENAI_ANALYST;
GRANT USE AI FUNCTIONS ON ACCOUNT TO ROLE GENAI_EMBED_RO;

-- GENAI_ADMIN: all Cortex capabilities
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER         TO ROLE GENAI_ADMIN;
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_ANALYST_USER TO ROLE GENAI_ADMIN;
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_AGENT_USER   TO ROLE GENAI_ADMIN;

-- GENAI_ANALYST: AI functions + Analyst
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER         TO ROLE GENAI_ANALYST;
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_ANALYST_USER TO ROLE GENAI_ANALYST;

-- GENAI_EMBED_RO: least privilege. CORTEX_EMBED_USER covers AI_EMBED, AI_MULTI_EMBED,
-- EMBED_TEXT_768 and EMBED_TEXT_1024.
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_EMBED_USER   TO ROLE GENAI_EMBED_RO;


In [ ]:
%%sql -r create_roles_2
-- If a role must never call AI_AGG / AI_SUMMARIZE_AGG, use AI_FUNCTIONS_USER instead of CORTEX_USER:
-- GRANT DATABASE ROLE SNOWFLAKE.AI_FUNCTIONS_USER TO ROLE SOME_SCALAR_ONLY_ROLE;

SHOW GRANTS TO ROLE GENAI_ANALYST;


---
## The function grant is not the data grant

This is the single most common support ticket in a new Cortex deployment. A role holds
`CORTEX_USER`, calls `AI_COMPLETE` on a literal string happily, and then fails the moment the prompt
reads from a table.

Think of the Cortex database role as a licence to operate the machine, and the object grants as the
key to the room the machine lives in. You need both, and they are issued by different people for
different reasons.

For an AI function over a table a role needs, on top of its Cortex role:

- `USAGE` on the database and on the schema
- `SELECT` on the table or view
- `USAGE` on the warehouse that runs the query
- `READ` on the stage, if the function reads files (`AI_PARSE_DOCUMENT`, `AI_EXTRACT`,
  `AI_TRANSCRIBE`); on an external stage the equivalent privilege is `USAGE`

→ [More on granting privileges](https://docs.snowflake.com/en/sql-reference/sql/grant-privilege)


In [ ]:
%%sql
-- Cortex roles cover the FUNCTIONS. Data access is always a separate set of grants.
USE DATABASE GENAI_STUDY;

GRANT USAGE  ON DATABASE GENAI_STUDY                        TO ROLE GENAI_ANALYST;
GRANT USAGE  ON SCHEMA   GENAI_STUDY.PUBLIC                 TO ROLE GENAI_ANALYST;
GRANT SELECT ON TABLE    GENAI_STUDY.PUBLIC.SUPPORT_TICKETS TO ROLE GENAI_ANALYST;
GRANT SELECT ON TABLE    GENAI_STUDY.PUBLIC.PRODUCTS        TO ROLE GENAI_ANALYST;
GRANT SELECT ON VIEW     GENAI_STUDY.PUBLIC.TICKETS_GDPR    TO ROLE GENAI_ANALYST;
GRANT USAGE  ON WAREHOUSE COMPUTE_WH                        TO ROLE GENAI_ANALYST;

-- A semantic view is its own object type, not a regular view.
GRANT SELECT ON SEMANTIC VIEW GENAI_STUDY.PUBLIC.PRODUCTS_SEMANTIC TO ROLE GENAI_ANALYST;


In [ ]:
%%sql -r data_grants_2
-- Querying the semantic view needs SELECT on the view itself; SELECT on its base tables is not
-- required for that. A YAML semantic model used by Cortex Analyst is different: that path needs
-- READ on the stage holding the YAML and SELECT on the tables the model names.

SHOW GRANTS TO ROLE GENAI_ANALYST;


---
## What each Cortex service adds on top

Every feature below needs its database role **and** its own object grants. The database role alone
gets you an error that mentions an object you thought you had access to.

| Feature | Database role (one of) | Object grants it also needs |
|---|---|---|
| **Cortex Search** — query | `CORTEX_USER` / `CORTEX_EMBED_USER` | `USAGE` on the service, and on its database and schema |
| **Cortex Search** — create | `CORTEX_USER` / `CORTEX_EMBED_USER` | `CREATE CORTEX SEARCH SERVICE` (or `OWNERSHIP`) on the schema, `SELECT` on the underlying tables or views, `USAGE` on the refreshing warehouse, and change tracking enabled on all underlying objects |
| **Cortex Analyst** | `CORTEX_USER` / `CORTEX_ANALYST_USER` | `READ` (or `WRITE`) on the stage holding a YAML semantic model, `SELECT` on the tables named in that model, and `USAGE` on any Cortex Search service the model references |
| **Cortex Agents** | `CORTEX_USER` / `CORTEX_AGENT_USER` | `CREATE AGENT` on the schema to create one; `USAGE` on the agent to call it; plus privileges on **every object each tool touches** |
| **Semantic view** as a query target | — | `SELECT` on the **semantic view** itself. You do *not* need `SELECT` on its base tables to query it — but you do need `SELECT` on them to *create* it, along with `CREATE SEMANTIC VIEW` and `USAGE` on the database and schema |

### Owner's rights, and why your schedule behaves differently from your session

A Cortex Search service, a stored procedure and a task all execute as their **owner**, not as the
caller. Two consequences worth memorising:

1. A pipeline that works when you run it by hand and silently produces nothing on a schedule is
   almost always a task-owner grant problem. Check what the task's owner role holds, not what you hold.
2. Two analysts querying the same search service see the same rows even if their table grants differ.
   That is the documented behaviour of an owner's-rights object, not a leak.

> **Agents inherit nothing.** `USAGE` on an agent lets you invoke it. It does not carry the semantic
> view, the search service or the warehouse along with it. An agent a user can call but whose tools
> they cannot reach answers nothing, and the error points at the tool.

→ [More on Cortex Search privileges](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview)
→ [More on Cortex Analyst privileges](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-analyst)


In [ ]:
%%sql
-- Object-level grants for Cortex services

-- Cortex Search service: USAGE to query it.
-- Remember services run with OWNER'S RIGHTS — the querying role sees whatever the OWNER can see,
-- so grant USAGE deliberately.
GRANT USAGE ON CORTEX SEARCH SERVICE GENAI_STUDY.PUBLIC.TICKET_SEARCH TO ROLE GENAI_ANALYST;

-- Stage access for AI_PARSE_DOCUMENT / AI_EXTRACT / AI_TRANSCRIBE on files
GRANT READ ON STAGE GENAI_STUDY.PUBLIC.DOCS_STAGE TO ROLE GENAI_ANALYST;


In [ ]:
%%sql -r service_grants_2
-- External stage: USAGE is the equivalent of READ on an internal stage.
-- FILE objects cannot be read from user stages, table stages, or fully client-side-encrypted stages.

-- Legacy YAML semantic models live on a stage — READ (or WRITE to edit) on that stage:
-- GRANT READ ON STAGE GENAI_STUDY.PUBLIC.SEMANTIC_MODELS TO ROLE GENAI_ANALYST;

-- Agent objects
-- GRANT USAGE ON AGENT GENAI_STUDY.PUBLIC.SUPPORT_AGENT TO ROLE GENAI_ANALYST;

SELECT CURRENT_ROLE() AS current_role;


In [ ]:
%%sql -r service_grants_3
SHOW GRANTS TO ROLE GENAI_ANALYST;


> ### ⚠️ Common misconceptions
>
> **"Querying a semantic view means I need `SELECT` on the tables behind it."**
> You need `SELECT` on the semantic view and nothing else — the docs are explicit that base-table
> `SELECT` is not required to query one. You do need it to *create* the view. Granting the base tables
> "to be safe" quietly widens access far beyond what the semantic view exposes.
> → [Querying semantic views](https://docs.snowflake.com/en/user-guide/views-semantic/querying)
>
> **"A semantic view is just a view, so `GRANT SELECT ON VIEW` works."**
> It is its own object type. The statement is `GRANT SELECT ON SEMANTIC VIEW <name> TO ROLE <r>;`.
> The wrong form fails with an object-does-not-exist error that sends people hunting for a typo in the
> name.
> → [Manage semantic views with SQL](https://docs.snowflake.com/en/user-guide/views-semantic/sql)
>
> **"Granting `USAGE` on the agent is enough for the user to get answers."**
> The agent runs its tools against the user's privileges. Without `SELECT` on the semantic view,
> `USAGE` on the search service and `USAGE` on the warehouse, the agent loads and every question comes
> back as a tool error — which reads like the agent is broken.
> → [Create and manage agents](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-manage)
>
> **"My grants are right, so the scheduled job will work."**
> Tasks run as their owner. If you granted yourself the access and the task is owned by another role,
> the schedule sees a different privilege set than your session did. The job usually does not error
> loudly — it returns nothing.
> → [Cortex Search overview](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview)


---
## Revoking, and proving it

Revocation is the other half of least privilege, and it is worth being deliberate about the order.
Revoke the narrowest thing that achieves the goal: pulling `CORTEX_USER` from a role removes every
Cortex capability including services, while pulling a single `USE AI FUNCTION` grant removes one
function and leaves the rest.

Two ways to check your work:

- `SHOW GRANTS TO ROLE <r>` — live, immediate, one role at a time
- `SNOWFLAKE.ACCOUNT_USAGE.GRANTS_TO_ROLES` — queryable across the whole account, with `DELETED_ON`
  marking revoked grants. Latency on this view can be up to 120 minutes, so a revoke you made a minute
  ago will still look active there. Use `SHOW GRANTS` when you need the truth right now.

→ [More on GRANTS_TO_ROLES](https://docs.snowflake.com/en/sql-reference/account-usage/grants_to_roles)


In [ ]:
%%sql
-- Revoke patterns — removing access

-- Revoke a Cortex database role
REVOKE DATABASE ROLE SNOWFLAKE.CORTEX_AGENT_USER FROM ROLE GENAI_ANALYST;

-- Revoke data access
REVOKE SELECT ON TABLE GENAI_STUDY.PUBLIC.SUPPORT_TICKETS FROM ROLE GENAI_EMBED_RO;


In [ ]:
%%sql -r revoke_examples_2
-- Revoke service access
-- REVOKE USAGE ON CORTEX SEARCH SERVICE GENAI_STUDY.PUBLIC.TICKET_SEARCH FROM ROLE GENAI_EMBED_RO;

-- Show what GENAI_ANALYST still has after revocation
SHOW GRANTS TO ROLE GENAI_ANALYST;


> ### 🤔 Stop and think
>
> - `CORTEX_USER` is one grant that covers everything; `AI_FUNCTIONS_USER` plus a handful of
>   `USE AI FUNCTION` grants is tighter but longer. Six months from now, which one will your team
>   actually keep accurate — and what is the cost of the one that drifts?
> - Owner's rights mean a Cortex Search service shows every caller the same rows. If your source table
>   has row-level restrictions, what are your options, and what does each cost in maintenance and in
>   index spend?
> - An agent needs privileges on every tool it calls. That means adding a tool to an agent silently
>   breaks it for every user who lacks the new tool's grants. Where in your process would you catch
>   that before the users do?


---
## From role to user

A role does nothing until someone holds it. Two properties on the user matter more than people
expect, because neither of them shows up in `SHOW GRANTS`:

- `DEFAULT_ROLE` — Cortex Agents determine a request's privileges from the **querying user's default
  role**, not from whatever role happens to be active in a session.
- `DEFAULT_WAREHOUSE` — that default role needs `USAGE` on it.

Get either wrong and the agent experience fails for a user whose grants look perfect in an audit.

→ [More on creating agents and their access requirements](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-manage)


In [ ]:
%%sql
-- Role hierarchy — assign functional roles to users
CREATE USER IF NOT EXISTS DATA_ANALYST_USER
    PASSWORD      = 'TempPass123!'
    DEFAULT_ROLE  = GENAI_ANALYST
    MUST_CHANGE_PASSWORD = TRUE;

GRANT ROLE GENAI_ANALYST TO USER DATA_ANALYST_USER;


In [ ]:
%%sql -r user_role_chain_2
-- Verify the full role + privilege chain
SHOW GRANTS TO USER DATA_ANALYST_USER;


In [ ]:
%%sql -r user_role_chain_3
SHOW GRANTS TO ROLE GENAI_ANALYST;


In [ ]:
%%sql -r user_role_chain_4
-- Account-wide audit of Cortex-related grants.
-- Latency on this view can be up to 120 minutes — use SHOW GRANTS to confirm a change you just made.
SELECT GRANTEE_NAME, PRIVILEGE, GRANTED_ON, NAME
FROM SNOWFLAKE.ACCOUNT_USAGE.GRANTS_TO_ROLES
WHERE NAME LIKE 'CORTEX%'
  AND DELETED_ON IS NULL
ORDER BY GRANTEE_NAME;


---
## Quick reference: which grant for which task?

| Task | Required |
|---|---|
| Call any AI function | `USE AI FUNCTIONS` (or `USE AI FUNCTION <name>`) **+** `CORTEX_USER` **or** `AI_FUNCTIONS_USER` |
| Call `AI_AGG` / `AI_SUMMARIZE_AGG` | `CORTEX_USER` — `AI_FUNCTIONS_USER` does **not** cover them |
| Call `AI_EMBED` / `AI_MULTI_EMBED` | `CORTEX_USER` **or** `CORTEX_EMBED_USER` |
| Use Cortex Analyst | `CORTEX_USER` **or** `CORTEX_ANALYST_USER`, plus stage `READ` for a YAML model and `SELECT` on the tables it names |
| Use Cortex Agents | `CORTEX_USER` **or** `CORTEX_AGENT_USER`, plus `USAGE` on the agent and privileges on every tool object |
| Query a Cortex Search service | `USAGE` on the service **and** on its database and schema |
| Create a Cortex Search service | `CORTEX_USER`/`CORTEX_EMBED_USER` + `CREATE CORTEX SEARCH SERVICE` on the schema + `SELECT` on base objects + `USAGE` on the refreshing warehouse + change tracking enabled |
| Query a semantic view | `SELECT` on the semantic view — base-table `SELECT` not required |
| Create a semantic view | `CREATE SEMANTIC VIEW` on the schema + `USAGE` on database and schema + `SELECT` on the underlying tables |
| Use only the Cortex REST API | `CORTEX_REST_API_USER` |
| Set `CORTEX_MODELS_ALLOWLIST` or `CORTEX_ENABLED_CROSS_REGION` | `ACCOUNTADMIN` |
| Grant a Cortex database role | `ACCOUNTADMIN` or `MANAGE GRANTS` |

---
## Worked scenario: three groups, three privilege sets

**The situation.** Data scientists need everything including agents. BI analysts need Cortex Analyst
but not agents. An ETL service account needs embeddings only, for a nightly vector refresh.

```sql
USE ROLE ACCOUNTADMIN;

-- Everyone needs the account privilege
GRANT USE AI FUNCTIONS ON ACCOUNT TO ROLE DATA_SCIENTIST_ROLE;
GRANT USE AI FUNCTIONS ON ACCOUNT TO ROLE BI_ANALYST_ROLE;

-- Data scientists
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER         TO ROLE DATA_SCIENTIST_ROLE;
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_ANALYST_USER TO ROLE DATA_SCIENTIST_ROLE;
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_AGENT_USER   TO ROLE DATA_SCIENTIST_ROLE;

-- BI analysts: no agents
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER         TO ROLE BI_ANALYST_ROLE;
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_ANALYST_USER TO ROLE BI_ANALYST_ROLE;

-- ETL service account: one function, one role, nothing else
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_EMBED_USER   TO ROLE ETL_SERVICE_ROLE;
GRANT USE AI FUNCTION AI_EMBED ON ACCOUNT         TO ROLE ETL_SERVICE_ROLE;
```

**What the tight ETL grant costs you:** the day someone wants that job to also summarise a column,
it fails, and the fix needs a privileged human. That is the point of least privilege, but budget for
the request queue it creates.

> **Least-privilege reminder.** `CORTEX_USER`, `COPILOT_USER` and `USE AI FUNCTIONS` are granted to
> `PUBLIC` by default. Until you revoke them from `PUBLIC`, every role design above is decorative —
> the users already had the access.
> ```sql
> REVOKE DATABASE ROLE SNOWFLAKE.CORTEX_USER  FROM ROLE PUBLIC;
> REVOKE DATABASE ROLE SNOWFLAKE.COPILOT_USER FROM ROLE PUBLIC;
> REVOKE USE AI FUNCTIONS ON ACCOUNT          FROM ROLE PUBLIC;
> ```


---
## Agents, in a little more detail

An **agent** is a Snowflake object that plans a response and calls tools — a Cortex Analyst semantic
view, a Cortex Search service, a custom function — to produce it. `CREATE AGENT` on the schema
creates one; `USAGE` on the agent lets a role call it.

```sql
CREATE OR REPLACE AGENT db.sch.support_agent
  COMMENT = 'Answers support questions over tickets and the product catalog'
  PROFILE = '{"display_name": "Support Analyst"}'
  FROM SPECIFICATION $$
    models:        { orchestration: "auto" }
    orchestration: { budget: { seconds: 120, tokens: 40000 } }
    instructions:  { response: "...", orchestration: "...", sample_questions: ["..."] }
    tools:         [ { tool_spec: { type: "cortex_analyst_text_to_sql", name: "sales"   } },
                     { tool_spec: { type: "cortex_search",              name: "tickets" } } ]
    tool_resources: { sales:   { semantic_view: "db.sch.products_semantic" },
                      tickets: { name: "db.sch.ticket_search" } }
  $$;

ALTER AGENT db.sch.support_agent MODIFY LIVE VERSION SET SPECIFICATION = $$ ... $$;
DESCRIBE AGENT db.sch.support_agent;
```

Two fields worth knowing by name:

- `instructions.sample_questions` is where an agent's suggested questions live. Cortex Analyst carries
  its own in the semantic model instead.
- `orchestration.budget` caps a single run with `seconds` and `tokens`. The token limit covers
  orchestration and excludes tokens spent inside tools such as Cortex Analyst or Cortex Search — so a
  tight orchestration budget does not cap total agent spend. Resource budgets on the agent object and
  per-user credit quotas are the other two documented controls; 3.3 covers the cost side.

→ [More on creating and managing agents](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-manage)


In [ ]:
%%sql
-- Full grant set for a Cortex Agents user
USE ROLE ACCOUNTADMIN;
CREATE ROLE IF NOT EXISTS GENAI_AGENT_USER;

GRANT USE AI FUNCTIONS ON ACCOUNT               TO ROLE GENAI_AGENT_USER;
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_AGENT_USER TO ROLE GENAI_AGENT_USER;

-- the agent object itself
GRANT USAGE ON AGENT GENAI_STUDY.PUBLIC.SUPPORT_AGENT TO ROLE GENAI_AGENT_USER;

-- every object the agent's tools touch — agents carry no privileges of their own
GRANT USAGE  ON DATABASE GENAI_STUDY                                    TO ROLE GENAI_AGENT_USER;
GRANT USAGE  ON SCHEMA   GENAI_STUDY.PUBLIC                             TO ROLE GENAI_AGENT_USER;
GRANT SELECT ON SEMANTIC VIEW GENAI_STUDY.PUBLIC.PRODUCTS_SEMANTIC      TO ROLE GENAI_AGENT_USER;
GRANT USAGE  ON CORTEX SEARCH SERVICE GENAI_STUDY.PUBLIC.TICKET_SEARCH  TO ROLE GENAI_AGENT_USER;
GRANT USAGE  ON WAREHOUSE COMPUTE_WH                                    TO ROLE GENAI_AGENT_USER;

-- An agent takes its privileges from the querying user's DEFAULT role, and that role needs
-- USAGE on the user's default warehouse.
ALTER USER DATA_ANALYST_USER SET DEFAULT_ROLE      = GENAI_AGENT_USER;
ALTER USER DATA_ANALYST_USER SET DEFAULT_WAREHOUSE = COMPUTE_WH;


In [ ]:
%%sql -r agents_list
-- Which agent objects does this account expose?
SHOW AGENTS IN ACCOUNT;

In [ ]:
%%sql -r agent_role_grants
-- Verify the effective grant set for the agent role
SHOW GRANTS TO ROLE GENAI_AGENT_USER;

---

## Check your understanding

Twelve questions on this notebook. Answer before expanding.

**1.** Which Cortex database roles are granted to `PUBLIC` by default?

<details><summary>Show answer</summary>

`SNOWFLAKE.CORTEX_USER` and `SNOWFLAKE.COPILOT_USER`. The account privilege `USE AI FUNCTIONS` is also
granted to `PUBLIC` by default. Everything else — `AI_FUNCTIONS_USER`, `CORTEX_EMBED_USER`,
`CORTEX_ANALYST_USER`, `CORTEX_AGENT_USER`, `CORTEX_REST_API_USER` — has to be granted explicitly.

→ [SNOWFLAKE database roles](https://docs.snowflake.com/en/sql-reference/snowflake-db-roles)

</details>

**2.** Which role is required to grant a Cortex database role to one of your account roles?

<details><summary>Show answer</summary>

`ACCOUNTADMIN`, or any role holding the `MANAGE GRANTS` privilege. A role that merely *holds*
`CORTEX_USER` cannot pass it on — holding a database role and being able to grant it are separate
things.

→ [SNOWFLAKE database roles](https://docs.snowflake.com/en/sql-reference/snowflake-db-roles)

</details>

**3.** What does `SNOWFLAKE.CORTEX_REST_API_USER` grant that the other roles do not?

<details><summary>Show answer</summary>

Use of the Cortex REST API without granting the rest of Cortex. It is the role for an application
service account whose only job is to call the API — the tightest option when you do not want that
account running AI functions in SQL as well.

→ [SNOWFLAKE database roles](https://docs.snowflake.com/en/sql-reference/snowflake-db-roles)

</details>

**4.** A role holds `CORTEX_USER` and `USE AI FUNCTIONS`. `SELECT AI_COMPLETE('llama3.1-8b', 'hi')`
works. `SELECT AI_COMPLETE('llama3.1-8b', ticket_text) FROM SUPPORT_TICKETS` fails. Why?

<details><summary>Show answer</summary>

The Cortex grants cover the function, not the data. The role is missing some combination of `USAGE`
on the database, `USAGE` on the schema, `SELECT` on the table and `USAGE` on the warehouse. The
literal-string call proved gates 1 and 2 pass — which is exactly why this failure confuses people.

→ [AISQL privileges and model access](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)

</details>

**5.** A role is granted `USAGE` on `SUPPORT_AGENT` and holds `CORTEX_AGENT_USER`. Every question
returns a tool error. What is missing?

<details><summary>Show answer</summary>

Privileges on the objects the agent's tools use: `SELECT` on the semantic view, `USAGE` on the Cortex
Search service, `USAGE` on the database, schema and warehouse. Agents run their tools against the
user's privileges and carry nothing of their own. Granting `CORTEX_USER` instead of
`CORTEX_AGENT_USER` would not help — that is a different gate.

→ [Create and manage agents](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-manage)

</details>

**6.** `GRANT SELECT ON VIEW GENAI_STUDY.PUBLIC.PRODUCTS_SEMANTIC TO ROLE BI_ANALYST;` fails. Fix it.

<details><summary>Show answer</summary>

A semantic view is its own object type:
`GRANT SELECT ON SEMANTIC VIEW GENAI_STUDY.PUBLIC.PRODUCTS_SEMANTIC TO ROLE BI_ANALYST;`. The failure
looks like a name problem, which sends people checking spelling rather than the object keyword. Note
you do not also need to grant the base tables for the role to query the semantic view.

→ [Querying semantic views](https://docs.snowflake.com/en/user-guide/views-semantic/querying)

</details>

**7.** A nightly task calls a Cortex Search service and writes nothing. Run by hand, the same SQL
returns rows. Where do you look first?

<details><summary>Show answer</summary>

At the **task owner's** grants. Tasks run as their owner, not as the person who created or triggered
them. The owner role probably lacks `USAGE` on the service or on the warehouse. Nothing errors
loudly, because an empty result is a perfectly valid outcome.

→ [Cortex Search overview](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview)

</details>

**8.** Creating a Cortex Search service fails even though the role holds `CORTEX_USER` and
`CREATE CORTEX SEARCH SERVICE` on the schema. Name two other requirements.

<details><summary>Show answer</summary>

`SELECT` on the underlying tables or views, `USAGE` on the warehouse that refreshes the service, and
change tracking enabled on all underlying objects. Change tracking is the one people forget — it is a
property of the source table, not a grant, so it never appears in `SHOW GRANTS`.

→ [Cortex Search overview](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview)

</details>

**9.** You revoked `CORTEX_USER` from a role two minutes ago, but
`ACCOUNT_USAGE.GRANTS_TO_ROLES` still shows the grant with `DELETED_ON` null. Is the revoke broken?

<details><summary>Show answer</summary>

No. `GRANTS_TO_ROLES` can lag by up to 120 minutes. `SHOW GRANTS TO ROLE <r>` reads live metadata and
is the right tool when you need to confirm a change you just made. Use the `ACCOUNT_USAGE` view for
account-wide audits, not for verifying a single statement.

→ [GRANTS_TO_ROLES](https://docs.snowflake.com/en/sql-reference/account-usage/grants_to_roles)

</details>

**10.** An ETL account needs only `AI_EMBED`. Compare granting `CORTEX_USER` + `USE AI FUNCTIONS`
against `CORTEX_EMBED_USER` + `USE AI FUNCTION AI_EMBED`. What does the tighter option cost?

<details><summary>Show answer</summary>

The tighter pair limits the blast radius of a compromised service account to one function on one
model set. The cost is friction: every future capability that job needs is a new grant, requiring a
privileged human, and the request will arrive at an inconvenient hour. Pick the tight version for
service accounts, where scope genuinely does not change; the broad version costs you far more when it
is the account that gets abused.

→ [AISQL privileges and model access](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)

</details>

**11.** Your compliance team wants a row-level restriction respected by a Cortex Search service. What
do you tell them?

<details><summary>Show answer</summary>

Search services run with **owner's rights**: every caller sees what the owner can see, so a caller's
own row-level grants are not applied at query time. If different audiences need different rows, build
separate services over separately filtered sources — which multiplies your serving cost, since each
service is charged per GB of indexed data per month. That trade-off is the honest answer, not a tuning
option.

→ [Cortex Search costs](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-costs)

</details>

**12.** Connecting to model access: a role holds every grant in this notebook and still gets an error
naming a specific model. Which gate is failing, and where is it configured?

<details><summary>Show answer</summary>

Gate 3, model access — covered in 3.1. The role needs either the model's application role
(`SNOWFLAKE."CORTEX-MODEL-ROLE-<MODEL>"`) or the model listed in `CORTEX_MODELS_ALLOWLIST`; the two
combine with OR. Neither is a grant on your own objects, so it never shows up while you audit
database and schema privileges.

→ [AISQL privileges and model access](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)

</details>
